# MiniROCKET side pipeline — scanning MiniROCKET features instead of PatchTST activations

Runs the **same** DeepScan, federated and privacy scripts as the main pipeline on MiniROCKET
features, with the same 355 PD/HC subjects, windows, folds, held-out HC reference and N = 10 bins.
Only the representation changes. The last cell compares the two encoders and applies a decision
rule fixed in `minirocket_scan/compare_models.py` before the results were seen.

**Prerequisite:** the main pipeline (`notebooks/000_full_pipeline.ipynb`) has been run, so the
PatchTST results and `minirocket_results.json` exist in `RESULTS_DIR`.

| Step | What | Time (approx.) |
|---|---|---|
| 0 | Setup: Drive, repo, packages | 2 min |
| 1 | Data: unzip PADS | 2 min |
| 2 | MiniROCKET features per fold | ~10 min |
| 3 | DeepScan Methods A/B/C | ~15 min (5,544 features vs PatchTST's 128) |
| 4 | Federated pseudo-site simulation | ~5 min |
| 5 | Privacy: re-identification sweep | ~15 min |
| 6 | Privacy: membership inference + DP | ~10 min |
| 7 | Comparison with PatchTST + decision | instant |
| 8 | Backup | 1 min |

MiniROCKET results go to `RESULTS_DIR/minirocket_scan/`; the PatchTST results are only read.
A GPU is not needed.

## 0. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/federated-subset-scanning-pads'
if not os.path.exists(REPO):
    !git clone https://github.com/isaacmutuma/federated-subset-scanning-pads.git {REPO}
else:
    !git -C {REPO} fetch origin && git -C {REPO} reset --hard origin/main
%cd {REPO}

# Optional: pin an exact commit for a reproducible paper run, e.g. PIN_COMMIT = 'abc1234'
PIN_COMMIT = None
if PIN_COMMIT:
    !git -C {REPO} checkout {PIN_COMMIT}
!git -C {REPO} log -1 --oneline

Mounted at /content/drive
Cloning into '/content/federated-subset-scanning-pads'...
remote: Enumerating objects: 408, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 408 (delta 87), reused 96 (delta 42), pack-reused 266 (from 1)
Receiving objects: 100% (408/408), 2.49 MiB | 16.02 MiB/s, done.
Resolving deltas: 100% (227/227), done.
/content/federated-subset-scanning-pads
733f1c7 (HEAD -> main, origin/main, origin/HEAD) MiniROCKET side pipeline: scan MiniROCKET features with the shared analysis scripts and compare to PatchTST


In [2]:
!pip install -q scipy scikit-learn adversarial-robustness-toolbox sktime
import sklearn, numpy as np, sktime
print('sklearn', sklearn.__version__, '| numpy', np.__version__, '| sktime', sktime.__version__)

sklearn 1.6.1 | numpy 2.1.3 | sktime 1.2.0


## 1. Data

In [3]:
import zipfile
ZIP_PATH     = '/content/drive/MyDrive/pads-parkisons-dataset-folder/pads-parkinsons-disease-smartwatch-dataset-1.0.0.zip'
EXTRACT_PATH = '/content/pads_dataset'
if not os.path.exists(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_PATH)
print(os.listdir(EXTRACT_PATH))

import sys
sys.path.insert(0, f'{REPO}/scripts')
import config as C
RESULTS_DIR = C.RESULTS_DIR                                   # PatchTST results (read only)
MR_DIR      = os.path.join(RESULTS_DIR, 'minirocket_scan')    # MiniROCKET results (Drive)
MR_ACT      = '/content/minirocket_scan/activations'        # MiniROCKET features (local disk, ~2.2 GB)
os.makedirs(MR_DIR, exist_ok=True)
print('PatchTST results:  ', RESULTS_DIR)
print('MiniROCKET results:', MR_DIR)
for f in ['deepscan_results_layer_2.json', 'minirocket_results.json', 'privacy_membership_layer_2.json']:
    print(f"  {'found  ' if os.path.exists(os.path.join(RESULTS_DIR, f)) else 'MISSING'} {f}")

['pads-parkinsons-disease-smartwatch-dataset-1.0.0']
Config loaded — ENV=colab
  PADS root:   /content/pads_dataset
  Checkpoints: /content/drive/MyDrive/patchtst_checkpoints_nw5
  Results:     /content/drive/MyDrive/patchtst_checkpoints_nw5
PatchTST results:   /content/drive/MyDrive/patchtst_checkpoints_nw5
MiniROCKET results: /content/drive/MyDrive/patchtst_checkpoints_nw5/minirocket_scan
  found   deepscan_results_layer_2.json
  found   minirocket_results.json
  found   privacy_membership_layer_2.json


## 2. MiniROCKET features
Fitted per fold on the training windows without labels (seeded), then applied to every split.
Features (5,544 per window) stay on the runtime's local disk; if the runtime restarts they are
rebuilt automatically. Folds that already exist are skipped; `FORCE_REEXTRACT = True` rebuilds them.

In [4]:
FORCE_REEXTRACT = False
!FORCE_REEXTRACT={int(FORCE_REEXTRACT)} MR_ACT_DIR={MR_ACT} python minirocket_scan/extract_minirocket_features.py 2>&1 | grep -E "Windows|HC=|FOLD|HC train|Features|Already|Saved|done|Error|Traceback" | tail -40

HC="Healthy"  PD="Parkinson's"
  Windows: (39050, 6, 200)  Skipped recordings: 0  Windows per subject: 110.0
  HC=8690  PD=30360
FOLD 1/5
HC train: 5,500  Val HC: 1,430  Val PD: 4,840  Test: 7,810
  Features per window: 5544
Saved fold 1
FOLD 2/5
HC train: 5,500  Val HC: 1,430  Val PD: 4,840  Test: 7,810
  Features per window: 5544
Saved fold 2
FOLD 3/5
HC train: 5,500  Val HC: 1,430  Val PD: 4,840  Test: 7,810
  Features per window: 5544
Saved fold 3
FOLD 4/5
HC train: 5,500  Val HC: 1,430  Val PD: 4,840  Test: 7,810
  Features per window: 5544
Saved fold 4
FOLD 5/5
HC train: 5,610  Val HC: 1,430  Val PD: 4,840  Test: 7,810
  Features per window: 5544
Saved fold 5
All folds done.


## 3. DeepScan on MiniROCKET features (same script and settings as PatchTST)

In [5]:
!ACT_DIR={MR_ACT} RESULTS_DIR={MR_DIR} LAYERS=minirocket python scripts/run_deepscan.py 2>&1 | grep -vE "^Fold [0-9] —|^fatal" | tail -40


Mean AUC:          0.7210 ± 0.0747
Mean Sensitivity:  0.5761 ± 0.0242
Mean Specificity:  0.7342 ± 0.0827
Mean Balanced Acc: 0.6551 ± 0.0534
Spec @ 90% sens:   0.2408 ± 0.1339
Pooled AUC 95% CI: {'auc': 0.7151, 'ci_low': 0.6555, 'ci_high': 0.7756}

B) Per-window FGSS

Mean AUC:          0.6869 ± 0.0715
Mean Sensitivity:  0.5653 ± 0.0247
Mean Specificity:  0.6958 ± 0.0840
Mean Balanced Acc: 0.6305 ± 0.0543
Spec @ 90% sens:   0.2275 ± 0.1082
Pooled AUC 95% CI: {'auc': 0.682, 'ci_low': 0.6193, 'ci_high': 0.7447}

C) Group FGSS

Mean AUC:          0.6435 ± 0.0962
Mean Sensitivity:  0.5436 ± 0.0338
Mean Specificity:  0.6192 ± 0.1156
Mean Balanced Acc: 0.5814 ± 0.0747
Spec @ 90% sens:   0.2258 ± 0.1448
Pooled AUC 95% CI: {'auc': 0.6378, 'ci_low': 0.5704, 'ci_high': 0.7074}

Paired A vs C (pooled): {'diff': 0.0773, 'ci_low': 0.0394, 'ci_high': 0.1148, 'p': 0.0}
FGSS failures (this layer): {'individ': 0, 'group': 0, 'individ_empty_subset': 0}

##################################################

## 4. Federated pseudo-site simulation

In [6]:
!ACT_DIR={MR_ACT} RESULTS_DIR={MR_DIR} LAYER=minirocket python scripts/run_federated_sim.py 2>&1 | grep -E "FOLD|Central|SUMMARY|transmits|Error|Traceback"

FOLD 1/5
Central AUC: 0.7386  Federated: 0.7386  Local-only: 0.7429 ± 0.0063
FOLD 2/5
Central AUC: 0.6114  Federated: 0.6114  Local-only: 0.6128 ± 0.0100
FOLD 3/5
Central AUC: 0.6750  Federated: 0.6750  Local-only: 0.6808 ± 0.0127
FOLD 4/5
Central AUC: 0.8432  Federated: 0.8432  Local-only: 0.8403 ± 0.0145
FOLD 5/5
Central AUC: 0.7286  Federated: 0.7286  Local-only: 0.7301 ± 0.0084
SUMMARY (mean, std over folds): {'central_auc': [0.7194, 0.0767], 'federated_auc_mean': [0.7194, 0.0767], 'local_auc_mean': [0.7214, 0.075]}
Each site transmits 221760 bytes of counts (5544 neurons x 10 bins)


## 5. Privacy — re-identification sweep

In [7]:
!ACT_DIR={MR_ACT} RESULTS_DIR={MR_DIR} LAYER=minirocket python scripts/privacy_analysis_bins.py 2>&1 | grep -E "Raw|N_BINS|ReID|DeepScan|Error|Traceback"

Raw activations: Top-1 0.3650 ± 0.0069  Top-5 0.6697 ± 0.0206  (chance Top-1 0.0199)
N_BINS = 5
  ReID Top-1 p-values: 0.2816 ± 0.0233  quantised raw: 0.3462 ± 0.0159
  DeepScan AUC (Method A): 0.7081 ± 0.0747
N_BINS = 10
  ReID Top-1 p-values: 0.3063 ± 0.0152  quantised raw: 0.3592 ± 0.0136
  DeepScan AUC (Method A): 0.7210 ± 0.0747
N_BINS = 20
  ReID Top-1 p-values: 0.3040 ± 0.0116  quantised raw: 0.3637 ± 0.0136
  DeepScan AUC (Method A): 0.7248 ± 0.0716
N_BINS = 50
  ReID Top-1 p-values: 0.3081 ± 0.0168  quantised raw: 0.3674 ± 0.0093
  DeepScan AUC (Method A): 0.7292 ± 0.0710
N_BINS = 100
  ReID Top-1 p-values: 0.3103 ± 0.0130  quantised raw: 0.3685 ± 0.0094
  DeepScan AUC (Method A): 0.7303 ± 0.0718


## 6. Privacy — membership inference on shared histograms (+ DP)

In [8]:
!ACT_DIR={MR_ACT} RESULTS_DIR={MR_DIR} LAYER=minirocket python scripts/privacy_membership.py 2>&1 | grep -E "FOLD|Membership|Method A|SUMMARY|Error|Traceback" | tail -40

FOLD 3/5
Membership AUC  site=  5  eps= None: 0.9819 ± 0.0262
Membership AUC  site=  5  eps= 10.0: 0.5008 ± 0.1585
Membership AUC  site=  5  eps=  1.0: 0.5243 ± 0.1345
Membership AUC  site= 10  eps= None: 0.9080 ± 0.0823
Membership AUC  site= 10  eps= 10.0: 0.5037 ± 0.0942
Membership AUC  site= 10  eps=  1.0: 0.5192 ± 0.1154
Membership AUC  site= 20  eps= None: 0.8612 ± 0.0949
Membership AUC  site= 20  eps= 10.0: 0.5007 ± 0.1044
Membership AUC  site= 20  eps=  1.0: 0.5058 ± 0.1098
Method A AUC  eps= None: 0.6750 ± 0.0000
Method A AUC  eps= 10.0: 0.6735 ± 0.0017
Method A AUC  eps=  1.0: 0.6733 ± 0.0015
FOLD 4/5
Membership AUC  site=  5  eps= None: 0.9800 ± 0.0290
Membership AUC  site=  5  eps= 10.0: 0.5011 ± 0.1436
Membership AUC  site=  5  eps=  1.0: 0.5284 ± 0.1366
Membership AUC  site= 10  eps= None: 0.9695 ± 0.0459
Membership AUC  site= 10  eps= 10.0: 0.5402 ± 0.1175
Membership AUC  site= 10  eps=  1.0: 0.5225 ± 0.0961
Membership AUC  site= 20  eps= None: 0.8926 ± 0.0957
Membership 

## 7. Comparison with PatchTST and decision
Paired on the same subjects. Decision rule: Method A AUC difference first; if its 95% CI includes 0,
then membership leakage, utility under ε = 1 noise, and histogram size, in that order.

In [9]:
!MR_RESULTS_DIR={MR_DIR} python minirocket_scan/compare_models.py

Config loaded — ENV=colab
  PADS root:   /content/pads_dataset
  Checkpoints: /content/drive/MyDrive/patchtst_checkpoints_nw5
  Results:     /content/drive/MyDrive/patchtst_checkpoints_nw5
=== Supervised classifier (subject-level, 5-fold) ===
  PatchTST     AUC 0.904 ± 0.054  spec@90%sens 0.737 ± 0.143
  MiniROCKET   AUC 0.913 ± 0.050  spec@90%sens 0.774 ± 0.121

=== DeepScan detection, PD vs HC (held-out HC reference, N=10) ===
  PatchTST (layer_2)  (commit e21f988)
     A: AUC 0.860 ± 0.067  pooled 0.854 [0.804, 0.899]  spec@90%sens 0.546
     B: AUC 0.851 ± 0.063  pooled 0.852 [0.806, 0.895]  spec@90%sens 0.547
     C: AUC 0.839 ± 0.065  pooled 0.837 [0.788, 0.882]  spec@90%sens 0.531
  MiniROCKET  (commit 733f1c7)
     A: AUC 0.721 ± 0.075  pooled 0.715 [0.655, 0.776]  spec@90%sens 0.241
     B: AUC 0.687 ± 0.071  pooled 0.682 [0.619, 0.745]  spec@90%sens 0.228
     C: AUC 0.643 ± 0.096  pooled 0.638 [0.570, 0.707]  spec@90%sens 0.226

=== Paired difference MiniROCKET − PatchTST (s

## 8. Backup

In [10]:
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
files = [f for f in os.listdir(MR_DIR) if f.endswith('.json')]
bdir = f'/content/drive/MyDrive/results_backup_minirocket_{stamp}'
os.makedirs(bdir, exist_ok=True)
for f in files:
    shutil.copy(os.path.join(MR_DIR, f), bdir)
print(f'Copied {len(files)} result files to {bdir}')

Copied 6 result files to /content/drive/MyDrive/results_backup_minirocket_20261025_1302
